# TPC_RP — Original Implementation

This notebook implements the **TypiClust** active learning algorithm on CIFAR-10, following:

> Hacohen, G., Dekel, O., & Weinshall, D. (2022).  
> **Active Learning on a Budget: Opposite Strategies Suit High and Low Budgets**.  
> *ICML 2022*. https://arxiv.org/abs/2202.02794

## Pipeline
1. Train SimCLR on the full CIFAR-10 training set (self-supervised)
2. Extract L2-normalised 512-dim backbone features
3. Run TypiClust active learning rounds
4. Evaluate with a linear probe after each round
5. Plot accuracy vs. label budget

## TypiClust algorithm recap
Each round:
- K = min(|labeled| + B, 500) clusters computed on ALL features
- Identify *uncovered* clusters (contain no labeled sample)
- Sort uncovered clusters by size (descending)
- From each of the B largest uncovered clusters, select the point with
  highest typicality: `Typicality(x) = (mean_{KNN} ||x - x_i||)^{-1}`
  where KNN is computed **within the cluster** with K_nn = min(20, cluster_size)

## 0. Setup

In [ ]:
# Install dependencies (uncomment in Colab)
# !pip install -q -r ../requirements.txt

import sys
sys.path.insert(0, '..')

import numpy as np
import torch

from src.utils import set_seed, load_cifar10, extract_features, save_results, plot_accuracy_curve, log
from src.simclr import SimCLRModel, train_simclr, load_simclr_model
from src.active_learning import TypiClust, run_active_learning_loop
from src.classifier import train_linear_probe, evaluate_linear_probe

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(SEED)
print(f'Device: {DEVICE}')

## 1. Hyperparameters

In [ ]:
# --- SimCLR (paper defaults) ---
SIMCLR_EPOCHS = 500    # use 100 for quick Colab tests
SIMCLR_BATCH  = 512
SIMCLR_LR     = 0.4   # SGD base LR for batch 512
SIMCLR_TEMP   = 0.5
CHECKPOINT    = '../results/simclr_checkpoint.pt'

# --- Active learning ---
INITIAL_BUDGET = 10    # labeled samples before round 1
QUERY_BUDGET   = 10    # samples queried per round
N_ROUNDS       = 20    # total rounds => 10 + 19*10 = 200 labeled samples

# --- TypiClust ---
MAX_CLUSTERS     = 500
MIN_CLUSTER_SIZE = 5

# --- Linear probe ---
PROBE_EPOCHS = 100
PROBE_LR     = 1e-3

## 2. Load Data

In [ ]:
train_dataset, test_dataset = load_cifar10(root='../data')
print(f'Train: {len(train_dataset):,}  |  Test: {len(test_dataset):,}')

## 3. SimCLR Pre-training

Checkpointed every 50 epochs to `results/simclr_checkpoint.pt`.  
Re-running this cell resumes from the last checkpoint automatically.

In [ ]:
model = SimCLRModel()
log('Starting SimCLR pre-training...')
model = train_simclr(
    dataset          = train_dataset,
    model            = model,
    num_epochs       = SIMCLR_EPOCHS,
    batch_size       = SIMCLR_BATCH,
    lr               = SIMCLR_LR,
    temperature      = SIMCLR_TEMP,
    checkpoint_path  = CHECKPOINT,
    checkpoint_every = 50,
    resume           = True,
    device           = DEVICE,
)
log('SimCLR pre-training complete.')

## 4. Feature Extraction

Extract **L2-normalised 512-dim** backbone features (before the projection head)  
and cache to disk — the AL loop reuses these without re-running the encoder.

In [ ]:
log('Extracting train features...')
train_feats, train_labels = extract_features(train_dataset, model, device=DEVICE)
log('Extracting test features...')
test_feats, test_labels = extract_features(test_dataset, model, device=DEVICE)

np.save('../results/train_features.npy', train_feats)
np.save('../results/train_labels.npy',   train_labels)
np.save('../results/test_features.npy',  test_feats)
np.save('../results/test_labels.npy',    test_labels)

print(f'Train features: {train_feats.shape}  |  Test features: {test_feats.shape}')
norms = np.linalg.norm(train_feats, axis=1)
print(f'Feature L2 norms — mean: {norms.mean():.4f}  min: {norms.min():.4f}  (should be ~1.0)')

## 5. Active Learning Loop (TypiClust)

In [ ]:
# TypiClust stores the full feature array; select() is called each round.
selector = TypiClust(
    features         = train_feats,
    max_clusters     = MAX_CLUSTERS,
    min_cluster_size = MIN_CLUSTER_SIZE,
    seed             = SEED,
)

def train_fn(labeled_idx, labels):
    return train_linear_probe(
        train_features = train_feats[labeled_idx],
        train_labels   = labels,
        epochs         = PROBE_EPOCHS,
        lr             = PROBE_LR,
        device         = DEVICE,
    )

def eval_fn(head):
    return evaluate_linear_probe(head, test_feats, test_labels, device=DEVICE)

history = run_active_learning_loop(
    features       = train_feats,
    labels         = train_labels,
    selector       = selector,
    initial_budget = INITIAL_BUDGET,
    query_budget   = QUERY_BUDGET,
    n_rounds       = N_ROUNDS,
    train_fn       = train_fn,
    eval_fn        = eval_fn,
    seed           = SEED,
)

save_results(history, '../results/typiclust_original_history.json')

## 6. Results

In [ ]:
import matplotlib.pyplot as plt

ax = plot_accuracy_curve(
    history['labelled_counts'],
    history['accuracies'],
    label='TypiClust',
    save_path='../plots/typiclust_original.png',
)
plt.show()

print('\nFinal accuracy: {:.2f}% with {:d} labeled samples'.format(
    history['accuracies'][-1] * 100,
    history['labelled_counts'][-1],
))